In [ ]:
!uv pip install --system "git+https://github.com/aqiq-marine/batch_colab_reaction.git"

In [ ]:
import os
print("Enter your Hugging Face access token. The input will be hidden for security.")
import getpass
HF_TOKEN = getpass.getpass("Hugging Face token:")

os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
!hf auth login --token $HF_TOKEN

In [ ]:
import os
import re
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# File uploader
# ============================================================

upload = widgets.FileUpload(
    accept=".xyz,.pdb",
    multiple=True,
    description="Upload reaction files"
)

output = widgets.Output()

display(upload)
display(output)


# ============================================================
# Validation
# ============================================================

def validate_files(change=None):

    with output:
        clear_output()

        if not upload.value:
            print("ファイルをアップロードしてください。")
            return

        filenames = list(upload.value.keys())

        print(f"{len(filenames)} files uploaded.")
        print()

        # ----------------------------------------------------
        # Allowed extensions
        # ----------------------------------------------------

        allowed_extensions = {".xyz", ".pdb"}

        for filename in filenames:

            suffix = Path(filename).suffix.lower()

            if suffix not in allowed_extensions:
                print(f"❌ Unsupported file format: {filename}")
                print("対応形式: .xyz, .pdb")
                return

        # ----------------------------------------------------
        # Parse filenames
        #
        # hoge_reactant.xyz
        # hoge_product.xyz
        # ----------------------------------------------------

        pattern = re.compile(
            r"^(.+)_(reactant|product)\.(xyz|pdb)$",
            re.IGNORECASE
        )

        reactions = {}

        for filename in filenames:

            match = pattern.match(filename)

            if match is None:
                print(f"❌ Invalid filename: {filename}")
                print(
                    "ファイル名は "
                    "hoge_reactant.xyz / hoge_product.xyz "
                    "の形式にしてください。"
                )
                return

            prefix = match.group(1)
            role = match.group(2).lower()
            extension = match.group(3).lower()

            if prefix not in reactions:
                reactions[prefix] = {}

            # Duplicate reactant/product
            if role in reactions[prefix]:

                print(
                    f"❌ Duplicate {role} file for reaction "
                    f"'{prefix}'"
                )
                return

            reactions[prefix][role] = {
                "filename": filename,
                "extension": extension,
            }

        # ----------------------------------------------------
        # Validate each reaction pair
        # ----------------------------------------------------

        reaction_pairs = []

        for prefix in sorted(reactions):

            files = reactions[prefix]

            # Reactant
            if "reactant" not in files:

                print(
                    f"❌ {prefix}: reactant file is missing."
                )
                return

            # Product
            if "product" not in files:

                print(
                    f"❌ {prefix}: product file is missing."
                )
                return

            reactant = files["reactant"]
            product = files["product"]

            # Same extension
            if reactant["extension"] != product["extension"]:

                print(
                    f"❌ {prefix}: "
                    "reactant/product の形式が異なります。"
                )

                print(
                    f"   reactant: "
                    f".{reactant['extension']}"
                )

                print(
                    f"   product : "
                    f".{product['extension']}"
                )

                return

            reaction_pairs.append({
                "name": prefix,
                "reactant": reactant["filename"],
                "product": product["filename"],
                "format": reactant["extension"],
            })

        # ----------------------------------------------------
        # Save uploaded files
        # ----------------------------------------------------

        for pair in reaction_pairs:

            for role in ["reactant", "product"]:

                filename = pair[role]

                data = upload.value[filename]["content"]

                with open(filename, "wb") as f:
                    f.write(data)

        # ----------------------------------------------------
        # Validation successful
        # ----------------------------------------------------

        print("✓ Validation successful")
        print()
        print(f"{len(reaction_pairs)} reaction pairs found.")
        print()

        for i, pair in enumerate(reaction_pairs, start=1):

            print(
                f"[{i}] {pair['name']}"
            )

            print(
                f"    Reactant: {pair['reactant']}"
            )

            print(
                f"    Product : {pair['product']}"
            )

            print()

        # Make available globally
        globals()["reaction_pairs"] = reaction_pairs


# ============================================================
# Observe upload
# ============================================================

upload.observe(
    validate_files,
    names="value"
)

In [ ]:
from fairchem.core import pretrained_mlip

model_name = "uma-s-1p2p1"

predictor = pretrained_mlip.get_predict_unit(
    model_name,
    device="cuda"
)

In [ ]:
from batch_colab_reaction import scan_reaction_files, run_all_reactions

pairs = scan_reaction_files(".")
results = run_all_reactions(pairs, predictor, output_dir="results")

In [ ]:
def print_summary(results) -> None:
    successful = [(name, result) for name, result in results.items() if "error" not in result]
    successful.sort(key=lambda item: item[1]["relative_energy"])
    print("\n" + "=" * 90)
    print("TS Energy Summary (sorted by ΔE‡)")
    print("=" * 90)
    print(f"{'Rank':>6}{'Reaction':<20}{'TS image':>10}{'TS energy (eV)':>20}{'ΔE‡ (eV)':>18}{'ΔE‡ (kcal/mol)':>20}")
    print("-" * 90)
    for rank, (name, result) in enumerate(successful, start=1):
        relative_energy = result["relative_energy"]
        print(f"{rank:>6}{name:<20}{result['ts_index']:>10d}{result['ts_energy']:>20.6f}{relative_energy:>18.6f}{relative_energy * 23.0605:>20.3f}")
    failed = [(name, result) for name, result in results.items() if "error" in result]
    if failed:
        print("\nFailed reactions:")
        for name, result in failed:
            print(f"  ✗ {name}: {type(result['error']).__name__}: {result['error']}")
    print("=" * 90)

print_summary(results)